# Dinka (Southwestern) topic classifier: live demo notebook
Loads the fine-tuned NLLB-encoder model from the Hugging Face Hub (`slickman-tyrus/dinka-sib200-classifier`) and serves it with Gradio.

**How to run:** `Runtime > Run all`. A GPU is optional (`Runtime > Change runtime type > T4 GPU` makes predictions faster). The last cell prints a **public link** (`https://....gradio.live`). Keep this notebook running while the link is in use; the link stops working when the session ends.

Why a notebook and not a hosted Space: current Hugging Face policy requires a paid plan for CPU-based Gradio Spaces, so this notebook is the free, reproducible way to run the app.

In [ ]:
!pip -q install gradio sentencepiece huggingface_hub

In [ ]:
import sys, torch
from huggingface_hub import snapshot_download

MODEL_REPO = "slickman-tyrus/dinka-sib200-classifier"
path = snapshot_download(MODEL_REPO)            # weights.pt, meta.json, model_def.py
sys.path.insert(0, path)
from model_def import load_predictor

predict = load_predictor(path)                  # uses the GPU if one is available
print("model loaded on", "GPU" if torch.cuda.is_available() else "CPU")

In [ ]:
EXAMPLES = [
    "Pan cɔl Turkey aci piu war adekdiet guɔ̈l piny ɣon kadiakic, Aegean tɔ ciem, Black sea tɔ ye nhom tueŋ ku Meditreanean tɔ ye cok piiny.",
    "Dek mɔ̈u amäth. Mɔ̈u akɔ̈c thöŋ gup tɛ̈ loi thok, ku ŋïcŋïc adan ye dek ake riil tör",
]
for t in EXAMPLES:                              # quick sanity check before launching the app
    p = predict([t])[0]
    print(sorted(p.items(), key=lambda kv: -kv[1])[:3], "\n  ", t[:80])

In [ ]:
import gradio as gr

def classify(text):
    text = (text or "").strip()
    if not text:
        return {}
    return predict([text])[0]

demo = gr.Interface(
    fn=classify,
    inputs=gr.Textbox(lines=4, label="Dinka sentence (Southwestern Dinka)"),
    outputs=gr.Label(num_top_classes=7, label="Predicted topic"),
    examples=[[e] for e in EXAMPLES],
    title="Dinka topic classifier",
    description=("Classifies a Dinka sentence into one of 7 topics (science/technology, travel, politics, sports, health, "
                 "entertainment, geography). Trained on SIB-200 (Southwestern Dinka, 701 translated Flores-200 sentences). "
                 "Limitations: small training set; other dialects such as Bor may be classified less reliably; "
                 "single sentences of news/travel-style text only."),
)
demo.launch(share=True, debug=False)           # prints a public link